In [ ]:
import os
import torch
import gc
import numpy as np
from universal_feature_extractor import UniversalFeatureExtractor
from utils import make_deterministic
from FeatureCache import FeatureCache
from cost_modules.sinkhorn_distance_estimator import SinkhornDistanceEstimator
from cost_modules.mean_cost_estimator import MeanCostEstimator
from cost_modules.centroid_distance_estimator import CentroidDistanceEstimator
from cost_modules.gaussian_mmd_estimator import GaussianMMDEstimator
from DatasetFeatureLoader import DatasetFeatureLoader
from models import Domain, DomainFeatures, BootstrapDraws, CostMatrix, ModelResult

SEED = 42
MODELS_TO_RUN = ["dinov2", "resnet50", "resnet18", "vgg16"]
USE_FEATURE_CACHE = True
# N_BOOTSTRAP is set from CT/T1/T2 — maximum samples available across all domains
N_BOOTSTRAP = 389
# Number of bootstrap draws per domain pair
N_DRAWS = 50
# For domains > N_BOOTSTRAP: random subset drawn with replacement
# For domains < N_BOOTSTRAP: whole dataset resampled with replacement
POOL_SIZE = 5 * N_BOOTSTRAP
OUTPUT_DIR = "/workspace/wasserstein-metric/pretraining_pipeline_results_v2/"

DOMAINS = [
    Domain("SAR-QXSLAB", "/workspace/mnt-data/QXSLAB_SAROPT/sar_256_oc_0.2"),
    Domain("OPT-QXSLAB", "/workspace/mnt-data/QXSLAB_SAROPT/opt_256_oc_0.2"),
    Domain("HE", "/workspace/mnt-data/HE/train"),
    Domain("IHC", "/workspace/mnt-data/IHC/train"),
    Domain("CT", "/workspace/mnt-data/CT/PNG"),
    Domain("T1", "/workspace/mnt-data/T1-MRI/PNG"),
    Domain("T2", "/workspace/mnt-data/T2-MRI/PNG"),
    Domain("SAR-SEN12", "/workspace/mnt-data/SEN12-splits/train/input"),
    Domain("OPT-SEN12", "/workspace/mnt-data/SEN12-splits/train/target"),
    Domain("ImageNet-1k", "/workspace/mnt-data/imagenet1k_flat"),
]

In [ ]:
# Prerequisites
os.makedirs(OUTPUT_DIR, exist_ok=True)
make_deterministic(SEED)

In [ ]:
# Allows caching of features
cache = None
if USE_FEATURE_CACHE:
    cache = FeatureCache(OUTPUT_DIR + "/.feature_cache")

In [ ]:
def make_cost_modules(model_name):
    return {
        "mean_cost": MeanCostEstimator(),
        "centroid_distance": CentroidDistanceEstimator(),
        "gaussian_mmd": GaussianMMDEstimator(),
        "sinkhorn_euclidean": SinkhornDistanceEstimator(
            cost="sqeuclidean", model=model_name
        ),
        "sinkhorn_cosine": SinkhornDistanceEstimator(cost="cosine", model=model_name),
    }

In [ ]:
def _load_all_features(extractor) -> list[DomainFeatures]:
    loader = DatasetFeatureLoader(
        extractor, pool_size=POOL_SIZE, seed=SEED, cache=cache
    )
    result = []
    for domain in DOMAINS:
        print(f"Loading: {domain.name}")
        features = loader.load(domain.path, domain.name)
        result.append(DomainFeatures(domain=domain, features=features))
    return result

In [ ]:
def _bootstrap_draws(domain_features: list[DomainFeatures]) -> BootstrapDraws:
    rng = np.random.default_rng(SEED)
    print("Drawing bootstrap samples...")
    draws = {
        df.domain.name: [
            df.features[rng.integers(0, len(df.features), N_BOOTSTRAP)]
            for _ in range(N_DRAWS)
        ]
        for df in domain_features
    }
    return BootstrapDraws(
        domain_features=domain_features,
        draws=draws,
        n_draws=N_DRAWS,
        n_bootstrap=N_BOOTSTRAP,
    )


def _run_for_model(model_name) -> ModelResult:
    extractor = None
    try:
        extractor = UniversalFeatureExtractor(model_name)
        draws = _bootstrap_draws(_load_all_features(extractor))
        result = ModelResult(model_name=model_name)
        cost_modules = make_cost_modules(model_name)
        for cost_name, cost_module in cost_modules.items():
            mean, std = cost_module.compute_cost_matrix(draws)
            result.cost_matrices.append(
                CostMatrix(
                    cost_name=cost_name,
                    model_name=model_name,
                    domains=draws.domains,
                    mean=mean,
                    std=std,
                )
            )
        return result
    finally:
        if extractor:
            del extractor
        torch.cuda.empty_cache()
        gc.collect()

In [ ]:
for model_name in MODELS_TO_RUN:
    model_result = _run_for_model(model_name)
    for cost_matrix in model_result.cost_matrices:
        cost_matrix.save(OUTPUT_DIR)
    print(f"Saved results for {model_name} to {OUTPUT_DIR}")